# Colab: KuaiRand ALS Gold + Baseline Training

This notebook is designed for Google Colab. It uses data stored in Google Drive, copies the required Parquet dataset to Colab local disk for faster Spark jobs, then builds the ALS Gold dataset and trains/evaluates:

1. Popularity baseline
2. Spark implicit ALS baseline

It logs metrics to MLflow under Drive and saves Gold artifacts/factors back to Drive.

Recommended runtime: CPU High-RAM if available. GPU is not required for Spark ML ALS.


## Expected Google Drive Layout

Upload this local folder to Drive:

```text
local: data/silver/kuairand/interactions/
```

Use this Drive path:

```text
MyDrive/recsys/data/silver/kuairand/interactions/
```

You do **not** need to upload raw CSV, bronze, users, videos, or old Gold artifacts for this ALS baseline notebook. The notebook will create:

```text
MyDrive/recsys/data/gold/als/v1_colab/
MyDrive/recsys/data/mlruns/
```


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
DRIVE_SILVER_INTERACTIONS = DRIVE_ROOT / 'data/silver/kuairand/interactions'
DRIVE_GOLD_DIR = DRIVE_ROOT / 'data/gold/als/v1_colab'
DRIVE_MLRUNS = DRIVE_ROOT / 'data/mlruns'

LOCAL_REPO = Path('/content/real-time-adaptive-recsys')
LOCAL_SILVER_DIR = LOCAL_REPO / 'data/silver/kuairand'
LOCAL_GOLD_DIR = LOCAL_REPO / 'data/gold/als/v1_colab'
LOCAL_MLRUNS = LOCAL_REPO / 'data/mlruns'

print('Drive interactions:', DRIVE_SILVER_INTERACTIONS)
print('Drive gold output:', DRIVE_GOLD_DIR)
print('Drive MLflow:', DRIVE_MLRUNS)

if not DRIVE_SILVER_INTERACTIONS.exists():
    raise FileNotFoundError(f'Missing Drive input: {DRIVE_SILVER_INTERACTIONS}')


## Clone Repo and Install Dependencies

This installs the project dependencies inside Colab. If you already cloned the repo in this runtime, the cell updates it.


In [ ]:
%%bash
set -e
if [ ! -d /content/real-time-adaptive-recsys/.git ]; then
  git clone https://github.com/kdnehihi/real-time-adaptive-recsys.git /content/real-time-adaptive-recsys
else
  cd /content/real-time-adaptive-recsys
  git pull
fi
cd /content/real-time-adaptive-recsys
pip install -q -r requirements.txt


## Copy Silver Interactions From Drive to Colab Local Disk

Spark runs much faster against `/content` than directly against mounted Drive. Outputs are copied back to Drive after training.


In [ ]:
%%bash
set -e
mkdir -p /content/real-time-adaptive-recsys/data/silver/kuairand
rm -rf /content/real-time-adaptive-recsys/data/silver/kuairand/interactions
cp -r /content/drive/MyDrive/recsys/data/silver/kuairand/interactions /content/real-time-adaptive-recsys/data/silver/kuairand/interactions

du -sh /content/real-time-adaptive-recsys/data/silver/kuairand/interactions


## Build Gold Dataset

This step creates chronological temporal split, train interactions, validation/test relevance, deterministic mappings, and manifest.

Default strength formula:

```text
interaction_strength = log1p(sum(
  0.5 * clip(watch_ratio, 0, 3)
+ 1.0 * long_view
+ 1.5 * is_like
+ 1.5 * is_comment
+ 1.5 * is_forward
+ 2.0 * is_follow
))
```

Optional: set `STRENGTH_TUNING_TRIALS = 10` to tune event-strength weights with Optuna/random-search proxy. For the first Colab run, keep it at 0.


In [ ]:
STRENGTH_TUNING_TRIALS = 0


In [ ]:
%%bash -s "$STRENGTH_TUNING_TRIALS"
set -e
TRIALS=$1
cd /content/real-time-adaptive-recsys
python scripts/build_kuairand_als_gold.py   --silver-dir data/silver/kuairand   --output-dir data/gold/als/v1_colab   --strength-tuning-trials "$TRIALS"   --overwrite


## Train and Evaluate Baselines

Recommended first full-ish Colab setting:

- rank grid: `32,64`
- regParam grid: `0.05,0.1`
- alpha grid: `10,20`
- maxIter: `5`

If runtime/disk is tight, use `ranks=16,32`, `maxIter=3`.


In [ ]:
RANKS = '32,64'
REG_PARAMS = '0.05,0.1'
ALPHAS = '10,20'
MAX_ITER = 5
ALS_OVER_GENERATE = 300
MAX_GRID_MODELS = 0  # 0 means run all combinations


In [ ]:
%%bash -s "$RANKS" "$REG_PARAMS" "$ALPHAS" "$MAX_ITER" "$ALS_OVER_GENERATE" "$MAX_GRID_MODELS"
set -e
RANKS=$1
REG_PARAMS=$2
ALPHAS=$3
MAX_ITER=$4
ALS_OVER_GENERATE=$5
MAX_GRID_MODELS=$6
cd /content/real-time-adaptive-recsys
python scripts/train_kuairand_als_baseline.py   --gold-dir data/gold/als/v1_colab   --tracking-dir data/mlruns   --ranks "$RANKS"   --reg-params "$REG_PARAMS"   --alphas "$ALPHAS"   --max-iter "$MAX_ITER"   --als-over-generate "$ALS_OVER_GENERATE"   --max-grid-models "$MAX_GRID_MODELS"   --run-name als_baseline_v1_colab   --overwrite


## Inspect Results


In [ ]:
import json
from pathlib import Path

manifest_path = LOCAL_GOLD_DIR / 'manifest.json'
eval_path = LOCAL_GOLD_DIR / 'evaluation_summary.json'
manifest = json.loads(manifest_path.read_text())
evaluation = json.loads(eval_path.read_text())

print('Temporal split')
print(json.dumps(manifest['temporal_split'], indent=2))
print('
Split summary')
print(json.dumps(manifest['split_summary'], indent=2))
print('
Evaluation summary')
print(json.dumps(evaluation, indent=2))


## Copy Gold Artifacts and MLflow Runs Back to Drive

This preserves outputs after the Colab runtime shuts down.


In [ ]:
%%bash
set -e
mkdir -p /content/drive/MyDrive/recsys/data/gold/als
rm -rf /content/drive/MyDrive/recsys/data/gold/als/v1_colab
cp -r /content/real-time-adaptive-recsys/data/gold/als/v1_colab /content/drive/MyDrive/recsys/data/gold/als/v1_colab

mkdir -p /content/drive/MyDrive/recsys/data
rm -rf /content/drive/MyDrive/recsys/data/mlruns
cp -r /content/real-time-adaptive-recsys/data/mlruns /content/drive/MyDrive/recsys/data/mlruns

du -sh /content/drive/MyDrive/recsys/data/gold/als/v1_colab /content/drive/MyDrive/recsys/data/mlruns


## Optional: Open MLflow UI

Colab does not expose ports by default. If you want MLflow UI, use the local files in Drive later, or run MLflow locally after downloading/copying `data/mlruns`.

Local command after syncing artifacts:

```bash
mlflow ui --backend-store-uri data/mlruns
```
